# Chapter 3: July 2025 Central TX Floods

## Overview

#### This notebook walks through how to access, visualize, and animate low-level composite reflectivity data from the Multi-Radar/Multi-Sensor (MRMS) system.

The case study focuses on the Central Texas flood event in July 2025, using reflectivity data hosted on AWS. The main steps include:
- Selecting and downloading MRMS data for specific timestamps
- Creating a static reflectivity map
- Building an animation to show reflectivity changes over time

This notebook is intended for students, forecasters, or researchers looking to explore radar visualization techniques or build familiarity with remote sensing workflows using Python.

### What is MRMS?

The Multi-Radar/Multi-Sensor (MRMS) system is a set of real-time analysis products developed by NOAA’s National Severe Storms Laboratory (NSSL). It brings together data from:
- Dozens of NEXRAD radars
- Surface observations
- Satellites
- Lightning detection networks

to create high-resolution snapshots of precipitation, severe weather, and related hazards.

MRMS updates every 2.5 minutes and is commonly used in operational forecasting, hydrology, aviation, and research.

---

### Goal of This Notebook

The goal of this notebook is to walk through a simple, practical workflow for visualizing radar reflectivity data using Python. Specifically, we’ll:

- Access MRMS Layer Composite Reflectivity Low data from AWS Open Data
- Plot a single reflectivity frame as a static map
- Animate an evenly spaced sequence from the morning of July 4, 2025, during the Central Texas flood event
- Demonstrate how to work with gridded radar data using open-source tools like MetPy, Cartopy, and xarray

## Imports
below are the python packages that are used for this code

In [ ]:
# Core packages
import gzip
import tempfile
import urllib.request  # For downloading MRMS .grib2.gz files
from datetime import datetime, timedelta
from io import StringIO

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmweather  # noqa: F401  (registers the ChaseSpectral colormap)
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt  # Plotting
import numpy as np
import numpy.ma as ma
import pandas as pd
import requests
import s3fs
import xarray as xr
from IPython.display import HTML  # To display the animation
from matplotlib.animation import ArtistAnimation, PillowWriter  # Animation
from metpy.plots import ctables  # For NWS reflectivity colormap
from scipy.interpolate import RegularGridInterpolator

## Define the Case, Domain, and Colormap

To build the animation, we'll use MRMS data from the morning of July 4, 2025, when
training thunderstorms over the Texas Hill Country drove the Guadalupe River's rapid
rise. MRMS writes a new file roughly every two minutes, so rather than guessing at
filenames we will ask AWS what is actually there and then pick an evenly spaced set
of frames from that list.

We also define the standard NWS reflectivity colormap using MetPy, which gives us
consistent color breaks every 5 dBZ -- a common setup for radar reflectivity plots.

### Access and Load MRMS Data

MRMS data is stored as `.grib2.gz` files on the AWS S3 public data bucket. Each file
holds a single timestamp of a single product.

Every download in this notebook follows the same three steps, so we write them once as
a helper function:

- **Fetch** the compressed file straight from AWS with `urllib.request.urlopen()`
- **Decompress** it with Python's built-in `gzip` module
- **Read** the GRIB2 bytes into an `xarray.DataArray` using the `cfgrib` engine

The helper also takes an optional bounding box. MRMS covers the whole CONUS on a 1-km
grid, so a single reflectivity field is about 100 MB in memory. Subsetting to our
region of interest as soon as the file is read keeps the notebook comfortable on a
laptop -- for the Texas domain below, it cuts each frame from ~98 MB to under 6 MB.

In [ ]:
def load_mrms(key, bbox=None):
    """Read one MRMS GRIB2 file from AWS into an xarray.DataArray.

    Parameters
    ----------
    key : str
        S3 key, e.g. ``noaa-mrms-pds/CONUS/<product>/<yyyymmdd>/<file>.grib2.gz``.
    bbox : tuple, optional
        ``(lon_min, lon_max, lat_min, lat_max)`` in degrees east/north, with
        longitudes given as negative west (e.g. ``-106``). When supplied, the
        field is subset before it is returned.
    """
    url = "https://noaa-mrms-pds.s3.amazonaws.com/" + key[len("noaa-mrms-pds/") :]
    with urllib.request.urlopen(url, timeout=60) as response:
        compressed_file = response.read()

    with tempfile.NamedTemporaryFile(suffix=".grib2") as f:
        f.write(gzip.decompress(compressed_file))
        f.flush()
        data = xr.load_dataarray(f.name, engine="cfgrib", decode_timedelta=True)

    if bbox is not None:
        lon_min, lon_max, lat_min, lat_max = bbox
        # MRMS latitudes run north-to-south, and longitudes are stored 0-360.
        data = data.sel(
            latitude=slice(lat_max, lat_min),
            longitude=slice(lon_min % 360, lon_max % 360),
        ).copy(deep=True)

    return data


# The domain we will plot throughout the reflectivity section
TX_BBOX = (-106, -93, 25, 36)

# One frame, to confirm the download path works before we animate
aws = s3fs.S3FileSystem(anon=True)
demo_key = (
    "noaa-mrms-pds/CONUS/LayerCompositeReflectivity_Low_00.50/20250704/"
    "MRMS_LayerCompositeReflectivity_Low_00.50_20250704-001040.grib2.gz"
)
data_in = load_mrms(demo_key, bbox=TX_BBOX)
data_in

## Set Up Reflectivity Colormap and Extract Data

This section gets the MRMS reflectivity data ready for plotting and builds a map to visualize it.

- **Colormap and Normalization:**  
  We use MetPy’s built-in NWSReflectivity colormap, which is designed for radar data in dBZ. The get_with_steps() function sets up color breaks every 5 dBZ — a common setup in operational radar displays.

- **Extract Coordinates and Data:**  
  We pull out the longitude, latitude, and reflectivity values from the data array. If the coordinates are in 1D (which happens in some MRMS products), we convert them to 2D using np.meshgrid() so they work with the plotting function.

- **Mask Low Reflectivity Values:**  
  Reflectivity values below 5 dBZ are masked out with ma.masked_where() to remove light noise and clutter from the map.

- **Set Up the Map:**  
  We create a static figure using matplotlib and Cartopy, with a PlateCarree projection centered over Texas. The domain is narrowed with set_extent() to focus on the region of interest.

- **Add Map Features:**  
  Coastlines, country borders, and U.S. state lines are added to give the plot geographic context.

- **Plot the Reflectivity:**  
  The reflectivity field is plotted using pcolormesh() with our defined colormap and normalization. A horizontal colorbar is added to show the dBZ scale.

- **Final Touches:**  
  We include a plot title and display the final figure with plt.show().

In [ ]:
refl_norm, refl_cmap = ctables.registry.get_with_steps("NWSReflectivity", 5, 5)

# 2. Extract coords & data
lons = data_in.longitude.values
lats = data_in.latitude.values
refl = data_in.values

# If coords are 1D, make them 2D
if lons.ndim == 1 and lats.ndim == 1:
    lons, lats = np.meshgrid(lons, lats)

In [ ]:
# 3. Plot
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-106, -93, 25, 36], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=1)
ax.add_feature(cfeature.STATES, linewidth=0.5)

mesh = ax.pcolormesh(
    lons,
    lats,
    ma.masked_where(refl < 5, refl),
    cmap=refl_cmap,
    norm=refl_norm,
    transform=ccrs.PlateCarree(),
)

cb = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.05, aspect=50)
cb.set_label("Reflectivity (dBZ)")

plt.title("MRMS Layer Composite Reflectivity – Texas", fontsize=14)
plt.show()

### Select Timestamps and Animate Reflectivity

Now we build an animation showing how low-level reflectivity evolved during the
flooding.

- **Ask AWS what exists:**
  MRMS file names end in the exact valid time, down to the second -- and that second
  drifts from scan to scan (`...-000040`, `...-000241`, `...-000442`). Constructing a
  file name from a round timestamp and testing whether it downloads therefore reports
  "missing" for data that is sitting right there: on July 4, 2025, only about a third
  of such guesses land on a real file. Instead we list the product's directory for the
  day, which returns all 720 available times, and select from that list.

- **Pick an even cadence:**
  `nearest_frames()` walks an evenly spaced grid of target times and, for each one,
  takes the closest file that actually exists. Because a scan is never more than a
  minute or so from any target, the result is a genuinely evenly spaced loop rather
  than whatever happened to line up.

- **Download and plot each frame:**
  For each selected time we load the file with `load_mrms()`, subsetting to the Texas
  domain on the way in, and draw it with `pcolormesh()` plus a title carrying the
  frame's UTC time. Each frame is stored for the animation.

- **Build and export the animation:**
  `ArtistAnimation` stitches the frames together; `plt.close(fig)` beforehand stops
  Jupyter from also showing a static copy. Finally we save the loop as a `.gif` with
  Pillow so it can be shared or embedded elsewhere.

In [ ]:
PRODUCT = "LayerCompositeReflectivity_Low_00.50"

# The window we want to animate, and how far apart the frames should be
window_start = datetime(2025, 7, 4, 0, 0)
window_end = datetime(2025, 7, 4, 6, 0)
cadence = timedelta(minutes=30)


def list_available_files(product, start, end):
    """Map every MRMS valid time on AWS to its S3 key, for the days spanned.

    MRMS file names carry the exact valid second, and that second drifts from
    scan to scan. Building a file name from a round timestamp therefore misses
    almost every file, so we list the bucket and let the archive tell us what
    is actually there.
    """
    available = {}
    day = start.date()
    while day <= end.date():
        prefix = f"noaa-mrms-pds/CONUS/{product}/{day:%Y%m%d}/"
        try:
            keys = aws.ls(prefix, refresh=True)
        except FileNotFoundError:
            keys = []
        for key in keys:
            stamp = key.split("_")[-1].removesuffix(".grib2.gz")
            available[datetime.strptime(stamp, "%Y%m%d-%H%M%S")] = key
        day += timedelta(days=1)
    return dict(sorted(available.items()))


def nearest_frames(available, start, end, step, tolerance=timedelta(minutes=5)):
    """Take the file closest to each target time on an evenly spaced grid."""
    frames = []
    target = start
    while target <= end:
        nearest = min(available, key=lambda valid: abs(valid - target))
        if abs(nearest - target) <= tolerance and nearest not in frames:
            frames.append(nearest)
        target += step
    return frames


available = list_available_files(PRODUCT, window_start, window_end)
frame_times = nearest_frames(available, window_start, window_end, cadence)

print(
    f"{len(available)} files on AWS for {window_start:%Y-%m-%d} "
    f"(one roughly every 2 minutes)\n"
)
print(
    f"Selected {len(frame_times)} frames at a {cadence.seconds // 60}-minute cadence:"
)
previous = None
for valid in frame_times:
    gap = (
        ""
        if previous is None
        else f"   (+{(valid - previous).total_seconds() / 60:.0f} min)"
    )
    print(f"  {valid:%Y-%m-%d %H:%M:%S} UTC{gap}")
    previous = valid

In [ ]:
# Set up colormap and normalization for reflectivity
refl_norm, refl_cmap = ctables.registry.get_with_steps("NWSReflectivity", 5, 5)

# Initialize animation container
frames = []

# Set up static map
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent(TX_BBOX, crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=1)
ax.add_feature(cfeature.STATES, linewidth=0.5)

# Loop through the selected times and collect frames
for valid in frame_times:
    print(f"Loading {valid:%H:%M:%S} UTC...")
    data_in = load_mrms(available[valid], bbox=TX_BBOX)

    # Extract coordinates and reflectivity data
    lons = data_in.longitude.values
    lats = data_in.latitude.values
    refl = data_in.values

    if lons.ndim == 1 and lats.ndim == 1:
        lons, lats = np.meshgrid(lons, lats)

    # Plot single frame (no show)
    mesh = ax.pcolormesh(
        lons,
        lats,
        ma.masked_where(refl < 5, refl),
        cmap=refl_cmap,
        norm=refl_norm,
        transform=ccrs.PlateCarree(),
    )

    # Create a title text that updates with each frame
    title = ax.text(
        0.5,
        1.02,
        f"MRMS Low-Level Reflectivity (dBZ) - {valid:%Y-%m-%d %H:%M} UTC",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=14,
    )

    # Save both mesh and title to animation frame
    frames.append([mesh, title])

# Create and display an animation
plt.close(fig)
anim = ArtistAnimation(fig, frames, interval=500, blit=True)
HTML(anim.to_jshtml())

In [ ]:
# Save animation as a .gif
anim.save("mrms_reflectivity_animation.gif", writer=PillowWriter(fps=2))

print("Animation saved as 'mrms_reflectivity_animation.gif'")

## Reflectivity Animation: Summary

We demonstrated how to access and animate low-level composite reflectivity data from the MRMS system using open-source Python tools. We focused on a short sequence from the July 4, 2025, Central Texas flood event to highlight how reflectivity features evolved.

This workflow is a flexible starting point for working with radar data, especially for case studies or quick visual diagnostics. The next section will continue building on this analysis with more approaches to explore the MRMS dataset!

## Comparison with ASOS Data

In [ ]:
# The 24-hour accumulation stamped 00Z on July 5 covers all of July 4 UTC
qpe_keys = aws.ls("noaa-mrms-pds/CONUS/RadarOnly_QPE_24H_00.00/20250705/")
qpe_key = next(k for k in qpe_keys if k.endswith("20250705-000000.grib2.gz"))

# Set lat and lon bounds
lat_min, lat_max = 28, 33
lon_min, lon_max = -102.5, -96.5

subset = load_mrms(qpe_key, bbox=(lon_min, lon_max, lat_min, lat_max))

url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

params = {
    "network": "TX_ASOS",  # Or just use "ASOS" for all U.S.
    "data": "p01i",
    # Routine hourly METARs only. ASOS also issues "special" reports between
    # them, and every report repeats the same past-hour total -- summing all of
    # them would count the same rain several times over.
    "report_type": "3",
    "year1": "2025",
    "month1": "7",
    "day1": "4",
    "year2": "2025",
    "month2": "7",
    "day2": "5",
    "format": "comma",
    "latlon": "yes",
}

# Make the request
response = requests.get(url, params=params)

# Parse CSV from response text
df = pd.read_csv(StringIO(response.text), skiprows=5)

# Drop missing precip values
df = df[df["p01i"] != "M"].copy()
df["p01i"] = df["p01i"].astype(float)

# Convert timestamp to datetime
df["valid"] = pd.to_datetime(df["valid"])

# Each p01i covers the hour *ending* at its timestamp, so the 24 reports from
# 01Z on the 4th through 00Z on the 5th line up with the MRMS accumulation above.
in_window = (df["valid"] > "2025-07-04 00:00") & (df["valid"] <= "2025-07-05 00:00")
df = df[in_window]

# Group by station and sum hourly precip
daily_precip = (
    df.groupby(["station", "lon", "lat"])["p01i"]
    .sum()
    .reset_index()
    .rename(columns={"p01i": "precip_in"})
)

daily_precip.sort_values("precip_in", ascending=False).head(10)

In [ ]:
# Set levels
levels = [
    0,
    0.01,
    0.1,
    0.25,
    0.50,
    1,
    1.5,
    2,
    2.5,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    12,
    14,
    16,
    18,
]

# Create a normalization object
cmap = plt.get_cmap("ChaseSpectral")  # Use full-resolution colormap
norm = mcolors.BoundaryNorm(levels, ncolors=cmap.N, clip=False)

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor="white")
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor="white")
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor="white")
# Add counties
ax.add_feature(
    cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_2_counties",
        scale="10m",
        facecolor="none",
        edgecolor="white",
        linewidth=0.3,
    )
)

mesh = ax.pcolormesh(
    subset.longitude,
    subset.latitude,
    subset / 25.4,  # Convert mm to inches
    norm=norm,
    cmap="ChaseSpectral",
    transform=ccrs.PlateCarree(),
)

# Overlay ASOS bubble plot
sc = ax.scatter(
    daily_precip["lon"],
    daily_precip["lat"],
    s=daily_precip["precip_in"] * 40,  # adjust bubble size scaling
    c=daily_precip["precip_in"],
    cmap=cmap,
    norm=norm,
    alpha=0.9,
    edgecolor="black",
    linewidth=0.4,
    transform=ccrs.PlateCarree(),
    zorder=10,
)

for size in [0.1, 0.5, 1.0, 2.0, 4.0]:
    ax.scatter(
        [],
        [],
        s=size * 40,
        c="gray",
        alpha=0.6,
        edgecolor="black",
        label=f'{size:.1f}"',
    )

ax.legend(scatterpoints=1, loc="lower right", title="ASOS Daily Rain", frameon=True)


cb = plt.colorbar(
    mesh, ax=ax, orientation="horizontal", pad=0.05, aspect=50, shrink=0.8
)
cb.set_label("Rainfall (in)")
# Add tick labels to colorbar
cb.set_ticks(levels)
cb.set_ticklabels([f"{level:.2f}" for level in levels])
cb.ax.tick_params(labelsize=10, rotation=45)

plt.title("MRMS 24-Hour Radar Only QPE vs. ASOS Stations (July 4, 2025)", fontsize=16)
plt.tight_layout()

In [ ]:
# Convert lat/lon coordinates from MRMS subset
lats = subset.latitude.values
lons = subset.longitude.values

# Ensure correct orientation (ascending order for interpolator)
if lats[0] > lats[-1]:
    lats = lats[::-1]
    subset = subset[::-1, :]

# Create interpolator (convert to inches)
interp_func = RegularGridInterpolator(
    (lats, lons), (subset / 25.4).values, bounds_error=False, fill_value=np.nan
)

# Convert ASOS longitude from degrees west to degrees east (0–360)
daily_precip["lon_east"] = daily_precip["lon"].apply(lambda x: x if x >= 0 else 360 + x)

station_coords = list(zip(daily_precip["lat"], daily_precip["lon_east"]))
daily_precip["mrms_in"] = interp_func(station_coords)


daily_precip["bias"] = daily_precip["precip_in"] - daily_precip["mrms_in"]

In [ ]:
# Define bias levels (nonlinear, symmetric)
bias_levels = [-20, -10, -5, -2, -1, -0.5, -0.1, 0, 0.1, 0.5, 1, 2, 5, 10, 20]

# Create BoundaryNorm
norm_bias = mcolors.BoundaryNorm(
    bias_levels, ncolors=plt.get_cmap("balance").N, clip=True
)


# Compute scatter sizes based on bias magnitude (optional scaling factor)
sizes = np.sqrt(np.abs(daily_precip["bias"])) * 150  # tweak 100 as needed

# Create figure and axis
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# Basemap features
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor="white")
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor="white")
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor="white")
ax.add_feature(
    cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_2_counties",
        scale="10m",
        facecolor="none",
        edgecolor="white",
        linewidth=0.3,
    )
)

# Pcolormesh for MRMS
mesh = ax.pcolormesh(
    subset.longitude,
    subset.latitude,
    subset / 25.4,  # Convert mm to inches
    norm=norm,
    cmap="ChaseSpectral",
    transform=ccrs.PlateCarree(),
)

sc = ax.scatter(
    daily_precip["lon"],
    daily_precip["lat"],
    c=daily_precip["bias"],
    s=sizes,
    cmap="balance",  # cmocean or any diverging colormap
    norm=norm_bias,
    edgecolor="black",
    linewidth=0.4,
    transform=ccrs.PlateCarree(),
    zorder=10,
)

# Add text labels for each station's bias
for _, row in daily_precip.iterrows():
    bias_val = row["bias"]
    if not np.isnan(bias_val):
        ax.text(
            row["lon"],
            row["lat"],
            f'{bias_val:.2f}"',
            fontsize=6,
            ha="center",
            va="center",
            transform=ccrs.PlateCarree(),
            zorder=11,
            color="white" if abs(bias_val) > 0.5 else "black",  # adjust for contrast
        )


# Bias colorbar (scatter)
cb1 = plt.colorbar(sc, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8, aspect=50)
cb1.set_label("ASOS - MRMS Bias (in)")
cb1.set_ticks(bias_levels)
cb1.ax.tick_params(labelsize=10)


# Add second colorbar (MRMS QPE from pcolormesh)
cb2 = plt.colorbar(mesh, ax=ax, orientation="vertical", pad=0.02, shrink=0.8)
cb2.set_label("MRMS QPE (in)")
cb2.ax.tick_params(labelsize=10)

# Title and layout
ax.set_title("ASOS vs. MRMS Radar-Only QPE Bias (July 4, 2025)", fontsize=16)
plt.tight_layout()

## Compare MRMS Radar-Only to Pass 1 and Pass 2 QPE

In [ ]:
def load_mrms_qpe_24h(
    date_str,
    product="RadarOnly_QPE_24H_00.00",
    lat_bounds=(28, 33),
    lon_bounds=(-102.5, -96.5),
):
    """Load and subset an MRMS 24-hour QPE field from AWS.

    Parameters
    ----------
    date_str : str
        Date in 'YYYYMMDD' format (e.g. '20250705'). The 00Z file for this date
        holds the accumulation over the *previous* 24 hours.
    product : str
        MRMS product folder (default: 'RadarOnly_QPE_24H_00.00')
    lat_bounds : tuple
        (lat_min, lat_max)
    lon_bounds : tuple
        (lon_min, lon_max), degrees east (negative for west)

    Returns
    -------
    xarray.DataArray
        Subset of the MRMS QPE field for the given domain and date.
    """
    prefix = f"noaa-mrms-pds/CONUS/{product}/{date_str}/"
    target = f"{date_str}-000000.grib2.gz"
    key = next((f for f in aws.ls(prefix) if f.endswith(target)), None)
    if key is None:
        raise FileNotFoundError(f"No {target} file found in {prefix}")

    lon_min, lon_max = lon_bounds
    lat_min, lat_max = lat_bounds
    return load_mrms(key, bbox=(lon_min, lon_max, lat_min, lat_max))

In [ ]:
subset_pass1 = load_mrms_qpe_24h("20250705", product="MultiSensor_QPE_24H_Pass1_00.00")
subset_pass2 = load_mrms_qpe_24h("20250705", product="MultiSensor_QPE_24H_Pass2_00.00")

In [ ]:
# Define levels and normalization
levels = [
    0,
    0.01,
    0.1,
    0.25,
    0.50,
    1,
    1.5,
    2,
    2.5,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    12,
    14,
    16,
    18,
]
cmap = plt.get_cmap("ChaseSpectral")
norm = mcolors.BoundaryNorm(levels, ncolors=cmap.N, clip=False)

# Create figure and subplots
fig, axes = plt.subplots(
    1, 3, figsize=(15, 6), subplot_kw={"projection": ccrs.PlateCarree()}
)

# Title mapping
titles = [
    "Radar-Only QPE",
    "Gauge-Corrected QPE (Pass 1)",
    "Gauge-Corrected QPE (Pass 2)",
]

# Loop through datasets and plot
for ax, data, title in zip(axes, [subset, subset_pass1, subset_pass2], titles):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # Basemap features
    ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor="white")
    ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor="white")
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor="white")
    ax.add_feature(
        cfeature.NaturalEarthFeature(
            category="cultural",
            name="admin_2_counties",
            scale="10m",
            facecolor="none",
            edgecolor="white",
            linewidth=0.3,
        )
    )

    # Pcolormesh plot
    mesh = ax.pcolormesh(
        data.longitude,
        data.latitude,
        data / 25.4,  # mm to inches
        norm=norm,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(title, fontsize=13)

# Shared colorbar below all plots
cb = fig.colorbar(
    mesh, ax=axes, orientation="horizontal", pad=0.08, aspect=50, shrink=0.8
)
cb.set_label("24-Hour Rainfall (in)", fontsize=12)
cb.set_ticks(levels)
cb.set_ticklabels([f"{level:.2f}" for level in levels])
cb.ax.tick_params(labelsize=10, rotation=45)

plt.suptitle("MRMS 24-Hour QPE Products (July 4, 2025)", fontsize=16)
plt.tight_layout(rect=[0, 0.25, 1, 0.98])  # leave space for suptitle and colorbar
plt.show()

In [ ]:
# Compute differences in inches
bias_pass1 = (subset_pass1 - subset) / 25.4
bias_pass2 = (subset_pass2 - subset) / 25.4

# Define nonlinear boundaries for bias (symmetric)
bias_levels = [-16, -10, -5, -2, -1, -0.5, -0.1, 0, 0.1, 0.5, 1, 2, 5, 10, 16]
norm_bias = mcolors.BoundaryNorm(
    bias_levels, ncolors=plt.get_cmap("balance").N, clip=True
)

# Create figure and subplots
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 6),
    constrained_layout=True,
    subplot_kw={"projection": ccrs.PlateCarree()},
)

# Titles
titles = ["Pass 1 – Radar-Only Bias (in)", "Pass 2 – Radar-Only Bias (in)"]

# Loop through plots
for ax, bias_data, title in zip(axes, [bias_pass1, bias_pass2], titles):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # Basemap features
    ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor="white")
    ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor="white")
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor="white")
    ax.add_feature(
        cfeature.NaturalEarthFeature(
            category="cultural",
            name="admin_2_counties",
            scale="10m",
            facecolor="none",
            edgecolor="white",
            linewidth=0.3,
        )
    )

    # Pcolormesh
    mesh = ax.pcolormesh(
        bias_data.longitude,
        bias_data.latitude,
        bias_data,
        cmap="balance",  # diverging colormap
        norm=norm_bias,
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(title, fontsize=13)

# Shared colorbar
cb = fig.colorbar(
    mesh, ax=axes, orientation="horizontal", pad=0.08, aspect=50, shrink=0.8
)
cb.set_label("Gauge-Corrected Bias from Radar-Only QPE (in)", fontsize=12)
cb.set_ticks(bias_levels)
cb.ax.tick_params(labelsize=10, rotation=45)

# Suptitle and layout
plt.suptitle("MRMS 24-Hour Gauge Correction Bias (July 4, 2025)", fontsize=16)
plt.show()